In [0]:
# Install Spark Kafka connector
%pip install confluent-kafka

In [0]:
dbutils.library.restartPython()

In [0]:
# Confluent Cloud config
BOOTSTRAP_SERVERS = "*****"
API_KEY = "*****"
API_SECRET = "*****"

In [0]:
from confluent_kafka import Consumer, TopicPartition, OFFSET_BEGINNING
import json
from datetime import datetime

consumer_config = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": API_KEY,
    "sasl.password": API_SECRET,
    "group.id": "databricks-cdc-consumer-reset",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": "false"
}

consumer = Consumer(consumer_config)

topics = [
    "banking.public.users",
    "banking.public.accounts",
    "banking.public.transactions"
]

# Manually assign multiple partitions per topic
partitions = []
for t in topics:
    for p in range(3):  # try partitions 0, 1, 2
        partitions.append(TopicPartition(t, p, OFFSET_BEGINNING))

consumer.assign(partitions)

users, accounts, transactions = [], [], []

print("Reading from Kafka...")

for _ in range(500):
    msg = consumer.poll(timeout=2.0)
    if msg is None:
        continue
    if msg.error():
        continue

    topic = msg.topic()
    event = json.loads(msg.value().decode("utf-8"))
    payload = event.get("payload", {})
    after = payload.get("after")
    op = payload.get("op")

    if not after or op not in ("c", "u", "r"):
        continue

    ingested_at = datetime.now().isoformat()

    if topic == "banking.public.users":
        users.append({
            "user_id": after.get("user_id"),
            "first_name": after.get("first_name"),
            "last_name": after.get("last_name"),
            "email": after.get("email"),
            "phone_number": after.get("phone_number"),
            "op": op,
            "ingested_at": ingested_at
        })
    elif topic == "banking.public.accounts":
        accounts.append({
            "account_id": after.get("account_id"),
            "user_id": after.get("user_id"),
            "account_type": after.get("account_type"),
            "balance": str(after.get("balance")),
            "status": after.get("status"),
            "op": op,
            "ingested_at": ingested_at
        })
    elif topic == "banking.public.transactions":
        transactions.append({
            "transaction_id": after.get("transaction_id"),
            "sender_account_id": after.get("sender_account_id"),
            "receiver_account_id": after.get("receiver_account_id"),
            "amount": str(after.get("amount")),
            "transaction_type": after.get("transaction_type"),
            "status": after.get("status"),
            "op": op,
            "ingested_at": ingested_at
        })

consumer.close()
print(f"Users: {len(users)}, Accounts: {len(accounts)}, Transactions: {len(transactions)}")

In [0]:
# Drop existing tables to avoid schema conflicts
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_users")
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_accounts")
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_transactions")

# Write users
if users:
    df_users = spark.createDataFrame(users)
    df_users.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_users")
    print(f"silver_users: {df_users.count()} records")

# Write accounts
if accounts:
    df_accounts = spark.createDataFrame(accounts)
    df_accounts.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_accounts")
    print(f"silver_accounts: {df_accounts.count()} records")

# Write transactions
if transactions:
    df_transactions = spark.createDataFrame(transactions)
    df_transactions.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_transactions")
    print(f"silver_transactions: {df_transactions.count()} records")

In [0]:
print("=== USERS ===")
display(spark.sql("SELECT * FROM workspace.default.silver_users LIMIT 5"))

In [0]:
print("=== ACCOUNTS ===")
display(spark.sql("SELECT * FROM workspace.default.silver_accounts LIMIT 5"))

In [0]:
print("=== TRANSACTIONS ===")
display(spark.sql("SELECT * FROM workspace.default.silver_transactions LIMIT 5"))

In [0]:
# Gold layer — business aggregations

# 1. Account summary per user
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_account_summary
    USING DELTA AS
    SELECT 
        u.user_id,
        u.first_name,
        u.last_name,
        u.email,
        COUNT(a.account_id) AS total_accounts,
        SUM(TRY_CAST(a.balance AS DOUBLE)) AS total_balance,
        MAX(a.account_type) AS account_type
    FROM workspace.default.silver_users u
    LEFT JOIN workspace.default.silver_accounts a ON u.user_id = a.user_id
    GROUP BY u.user_id, u.first_name, u.last_name, u.email
""")
print("gold_account_summary created")

# 2. Daily transaction volume
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_daily_transactions
    USING DELTA AS
    SELECT
        DATE(ingested_at) AS transaction_date,
        COUNT(transaction_id) AS total_transactions,
        SUM(TRY_CAST(amount AS DOUBLE)) AS total_volume,
        AVG(TRY_CAST(amount AS DOUBLE)) AS avg_transaction_amount,
        transaction_type
    FROM workspace.default.silver_transactions
    GROUP BY DATE(ingested_at), transaction_type
    ORDER BY transaction_date DESC
""")
print("gold_daily_transactions created")

# 3. Top senders
spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.gold_top_senders
    USING DELTA AS
    SELECT
        sender_account_id,
        COUNT(*) AS total_transactions_sent,
        SUM(TRY_CAST(amount AS DOUBLE)) AS total_amount_sent,
        AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount_sent
    FROM workspace.default.silver_transactions
    GROUP BY sender_account_id
    ORDER BY total_amount_sent DESC
    LIMIT 10
""")
print("gold_top_senders created")

In [0]:
print("=== ACCOUNT SUMMARY ===")
display(spark.sql("SELECT * FROM workspace.default.gold_account_summary LIMIT 5"))

In [0]:
print("=== DAILY TRANSACTIONS ===")
display(spark.sql("SELECT * FROM workspace.default.gold_daily_transactions"))

In [0]:
print("=== TOP SENDERS ===")
display(spark.sql("SELECT * FROM workspace.default.gold_top_senders"))